# RQ1 — presence-only method selection: true one-class vs. presence-background, sensitivity at fixed predicted area

**Purpose.** Chooses the presence-only classifier of the RQ1 curve, as
decided on 2026-11-17 after the supervisor's meeting
([`thesis/notes.md`](../../thesis/notes.md#rq1-presence-only)). Two
families, one protocol:

- **(a) True one-class** — trained on presences only, no background:
  Isolation Forest, One-Class SVM, PCA reconstruction error, a small
  autoencoder (the non-linear form of the PCA), and the
  autoencoder/PCA + Isolation Forest ensembles (what the supervisor
  named).
- **(b) Presence-background** — background enters training as an
  unlabelled sample of the landscape, never as absence: L2 logistic
  regression and MaxEnt (`elapid`), presences vs. training-block
  background.

Both families get a small hyperparameter sweep (section 3), because
the earlier runs of this notebook used defaults only; the final
comparison (section 4) uses the best configuration of each model.
Earlier runs also included a presence-vs-background Random Forest and
an RBF SVM with balanced class weights; both were removed on the same
day (the first is not presence-only, the second was my addition with
no source and tied with the logistic regression). Those runs and their
numbers are recorded only in `thesis/notes.md`, linked above.

**Why this notebook exists:**
[`one_class_background_free_evaluation.ipynb`](one_class_background_free_evaluation.ipynb)
showed that an evaluation using only presences cannot rank models.
Here the background is used **only to measure area**: no background
point is ever labelled as absence, so an unrecorded acacia in the
background never counts as an error.

**Scope:** citizen-science *A. dealbata* presences, TESSERA embeddings,
leave-one-block-out over the 15 blocks of 10 km with at least 5
presences. Expert arm and curve not run here.


## 1. Data

Presences joined to their TESSERA pilot embedding by pixel id (as in
the one-class notebooks); the uniform background pool, checked
against the manifest. Blocks as in
`one_class_background_free_evaluation.ipynb`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.background import load_background
from src.embeddings import block_id, load_pilot_embeddings, match_to_pixel

pilot_df, pilot_emb = load_pilot_embeddings(ROOT / "data" / "processed" / "embeddings" / "tessera_v1.1_2023_rq1_pilot")
dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
pres_x, pres_y, pres_emb, _ = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)
pres_block = block_id(pres_x, pres_y)

bg = load_background(ROOT, "uniform")
bg_emb, bg_block = bg["emb"], block_id(bg["x"], bg["y"])

block_counts = pd.Series(pres_block).value_counts()
folds = sorted(block_counts[block_counts >= 5].index)
bg_per_fold = pd.Series(bg_block).value_counts().reindex(folds).fillna(0).astype(int)
print(f"{len(pres_emb)} presences, {len(bg_emb)} background points, {len(folds)} folds")
print("presences per fold:", block_counts[folds].sort_values().tolist())
print("background points per fold:", bg_per_fold.sort_values().tolist())


## 2. Protocol and decision criterion (written before running)

**Folds:** leave-one-block-out; in each fold the test block's presences
and background are held out, everything else trains.

**Training:** (a) models fit on training presences only, scaled by a
`StandardScaler` fit on them; (b) models fit on training presences (1)
vs. training-block background (0), scaled on both.

**Metric — sensitivity at a fixed fraction of predicted area.** For
each held-out presence, the *area needed to include it* is the
fraction of its block's held-out background points with a score at or
above its own (the background is the sample of that block's
landscape). Pooling every held-out presence over the 15 folds gives
the curve "sensitivity vs. predicted area": at X% of area, the
fraction of presences whose needed area is ≤ X%. Rank-based within
each fold, so model scores need no common scale across folds. With
15–40 background points per block the per-presence resolution is
coarse (about 3%); pooling 2 464 presences smooths it.

**Primary number:** pooled sensitivity at **10% of area** — my choice,
not grounded in a source; the full 5–30% curve is reported so the
choice is visible. Secondary: per-fold sensitivity at 10%, mean and sd
over the 15 folds (blocks weighted equally).

**Sweep (section 3), fixed before running.** Every configuration runs
on the same folds; "best" means highest pooled sensitivity at 10%.

- (a) Isolation Forest: `max_samples` 256 (default) or 1 024,
  `max_features` 1.0 or 0.5; 200 trees. (`contamination` only shifts
  the score offset and cannot change a ranking, so it is not swept.)
- (a) One-Class SVM, RBF: `nu` 0.05 / 0.10 / 0.20 × `gamma` at 0.5 /
  1 / 2 times the `"scale"` value.
- (a) PCA reconstruction error at 90% variance (as before), and PCA to
  20 components followed by One-Class SVM (`nu` 0.10) or Isolation
  Forest (defaults), to test whether density estimation in fewer
  dimensions helps.
- (a) Autoencoder: `MLPRegressor` 128 → 32 → 128 (tanh), reconstruction
  error as score; z-score ensembles of Isolation Forest with the PCA
  and with the autoencoder (z-scores from the training presences).
- (b) Logistic regression L2, `class_weight="balanced"`, `C` 0.01 /
  0.1 / 1 / 10.
- (b) MaxEnt (`elapid.MaxentModel`, defaults otherwise): features
  linear or linear + quadratic × `beta_multiplier` 0.5 / 1.5 (default)
  / 4. Hinge and product features on 128 dimensions would create
  thousands of features for ~2 300 presences and are not run.

**Caveat declared in advance:** the best configuration is chosen on
the same test folds that report its number, for both families alike;
the sweep numbers are therefore optimistic by the same selection
effect. No nested cross-validation (15 folds of 7–819 presences would
not support it).

**Criteria (section 4):**

1. **One-class list.** The best configuration of each one-class model
   is kept for the final table. If no one-class configuration reaches
   **0.30** pooled sensitivity at 10% (chance 0.10), the one-class side
   of RQ1 closes with Isolation Forest and One-Class SVM only; PCA, the
   autoencoder and the ensembles are then recorded as tested and
   dropped.
2. **Classifier of the curve.** The best configuration of the logistic
   regression and of MaxEnt are compared; the higher pooled sensitivity
   at 10% wins, and if they are within **0.02** the logistic regression
   is kept as the simpler model.
3. The family comparison (every (b) vs. every (a)) is reported at every
   area fraction.


## 3. Sweep: every configuration, sorted by the primary number

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from elapid import MaxentModel

AREAS = np.arange(0.05, 0.31, 0.05)
PRIMARY_AREA = 0.10


def recon_score(model, x_s):
    return -((x_s - model.inverse_transform(model.transform(x_s))) ** 2).sum(axis=1)


class AE:
    """128 -> 32 -> 128 autoencoder; same interface as PCA for recon_score."""
    def __init__(self, seed=0):
        self.m = MLPRegressor(hidden_layer_sizes=(32,), activation="tanh", max_iter=500,
                              random_state=seed)
    def fit(self, x):
        self.m.fit(x, x); return self
    def transform(self, x):
        return x
    def inverse_transform(self, x):
        return self.m.predict(x)


def needed_area(pres_scores, bg_scores):
    # fraction of the block's background scoring at or above each presence
    return np.array([(bg_scores >= s).mean() for s in pres_scores])


def z(s, ref):
    return (s - ref.mean()) / ref.std()


IF_CFG = {"IF default": dict(), "IF max_samples=1024": dict(max_samples=1024),
          "IF max_features=0.5": dict(max_features=0.5)}
OCSVM_CFG = {f"OCSVM nu={nu} gamma=x{g}": (nu, g) for nu in (0.05, 0.10, 0.20) for g in (0.5, 1, 2)}
LR_CFG = {f"LR C={c}": c for c in (0.01, 0.1, 1, 10)}
MX_CFG = {f"MaxEnt {'+'.join(f[0] for f in ft)} beta={b}": (ft, b)
          for ft in (["linear"], ["linear", "quadratic"]) for b in (0.5, 1.5, 4)}
needed, per_fold = {}, {}


def record(name, sp, sb):
    na = needed_area(sp, sb)
    needed.setdefault(name, []).append(na)
    per_fold.setdefault(name, []).append((na <= PRIMARY_AREA).mean())


for tb in folds:
    tr_p, te_p = pres_emb[pres_block != tb], pres_emb[pres_block == tb]
    tr_b, te_b = bg_emb[bg_block != tb], bg_emb[bg_block == tb]

    # (a) presences only
    sc_a = StandardScaler().fit(tr_p)
    trp, tep, teb = sc_a.transform(tr_p), sc_a.transform(te_p), sc_a.transform(te_b)
    gamma_scale = 1.0 / (trp.shape[1] * trp.var())

    iso = {}
    for name, kw in IF_CFG.items():
        iso[name] = IsolationForest(n_estimators=200, random_state=42, **kw).fit(trp)
        record(f"(a) {name}", iso[name].decision_function(tep), iso[name].decision_function(teb))
    for name, (nu, g) in OCSVM_CFG.items():
        m = OneClassSVM(kernel="rbf", nu=nu, gamma=g * gamma_scale).fit(trp)
        record(f"(a) {name}", m.decision_function(tep), m.decision_function(teb))

    pca90 = PCA(n_components=0.90, svd_solver="full", random_state=0).fit(trp)
    record("(a) PCA recon (90% var)", recon_score(pca90, tep), recon_score(pca90, teb))
    pca20 = PCA(n_components=20, svd_solver="full", random_state=0).fit(trp)
    trp20, tep20, teb20 = pca20.transform(trp), pca20.transform(tep), pca20.transform(teb)
    m = OneClassSVM(kernel="rbf", nu=0.10, gamma="scale").fit(trp20)
    record("(a) PCA20 -> OCSVM", m.decision_function(tep20), m.decision_function(teb20))
    m = IsolationForest(n_estimators=200, random_state=42).fit(trp20)
    record("(a) PCA20 -> IF", m.decision_function(tep20), m.decision_function(teb20))

    ae = AE().fit(trp)
    record("(a) Autoencoder recon", recon_score(ae, tep), recon_score(ae, teb))

    iso_d = iso["IF default"]
    iso_tr = iso_d.decision_function(trp)
    for name, rec in (("PCA", pca90), ("AE", ae)):
        rec_tr = recon_score(rec, trp)
        ens = lambda x: (z(iso_d.decision_function(x), iso_tr) + z(recon_score(rec, x), rec_tr)) / 2
        record(f"(a) Ensemble {name}+IF", ens(tep), ens(teb))

    # (b) presences vs. training-block background (unlabelled)
    sc_b = StandardScaler().fit(np.vstack([tr_p, tr_b]))
    X_b = sc_b.transform(np.vstack([tr_p, tr_b])); y_b = np.r_[np.ones(len(tr_p)), np.zeros(len(tr_b))]
    tep_b, teb_b = sc_b.transform(te_p), sc_b.transform(te_b)
    for name, c in LR_CFG.items():
        m = LogisticRegression(C=c, class_weight="balanced", max_iter=5000).fit(X_b, y_b)
        record(f"(b) {name}", m.decision_function(tep_b), m.decision_function(teb_b))
    for name, (ft, b) in MX_CFG.items():
        m = MaxentModel(feature_types=ft, beta_multiplier=b, n_cpus=4, random_state=42).fit(X_b, y_b)
        record(f"(b) {name}", m.predict(tep_b), m.predict(teb_b))

MODELS = list(needed)
pooled = {m: np.concatenate(v) for m, v in needed.items()}
sweep = pd.DataFrame({
    f"pooled sens. @{int(PRIMARY_AREA*100)}%": {m: (pooled[m] <= PRIMARY_AREA).mean() for m in MODELS},
    "per-fold mean": {m: np.mean(per_fold[m]) for m in MODELS},
    "per-fold sd": {m: np.std(per_fold[m]) for m in MODELS},
})
print(f"{len(MODELS)} configurations")
sweep.round(3).sort_values(sweep.columns[0], ascending=False)


## 4. Final table (best configuration of each model) and decision

In [ ]:
# Best configuration per model (criterion of section 2), then the final table
GROUPS = {
    "(a) Isolation Forest": lambda m: m.startswith("(a) IF"),
    "(a) One-Class SVM": lambda m: m.startswith("(a) OCSVM"),
    "(a) PCA (recon or PCA20 ->)": lambda m: m.startswith("(a) PCA"),
    "(a) Autoencoder": lambda m: m == "(a) Autoencoder recon",
    "(a) Ensemble +IF": lambda m: m.startswith("(a) Ensemble"),
    "(b) Logistic regression": lambda m: m.startswith("(b) LR"),
    "(b) MaxEnt": lambda m: m.startswith("(b) MaxEnt"),
}
prim = sweep.columns[0]
best = {g: sweep[[f(m) for m in sweep.index]][prim].idxmax() for g, f in GROUPS.items()}
final = sweep.loc[list(best.values())].copy()
final.index = [f"{g}  [{b}]" for g, b in best.items()]
curve = pd.DataFrame({m: [(pooled[b] <= a).mean() for a in AREAS] for m, b in zip(final.index, best.values())},
                     index=[f"{int(a*100)}% area" for a in AREAS]).T

one_class_max = final[[i.startswith("(a)") for i in final.index]][prim].max()
lr_best, mx_best = final.loc[[i.startswith("(b) Logistic") for i in final.index], prim].iloc[0], \
                   final.loc[[i.startswith("(b) MaxEnt") for i in final.index], prim].iloc[0]
print(f"best one-class @10%: {one_class_max:.3f}  -> {'some one-class reaches 0.30' if one_class_max >= 0.30 else 'none reaches 0.30: one-class list closes with IF and OCSVM'}")
print(f"LR {lr_best:.3f} vs MaxEnt {mx_best:.3f} -> classifier: "
      f"{'logistic regression' if (lr_best >= mx_best) or (mx_best - lr_best < 0.02) else 'MaxEnt'}")
final.round(3)


In [ ]:
curve.round(3)

In [ ]:
pd.DataFrame({m: per_fold[b] for m, b in zip(final.index, best.values())}, index=folds).round(2)

## 5. Result and decision

28 configurations, one run (2026-11-17), seeds fixed. Pooled
sensitivity at 10% of predicted area (chance 0.10), best configuration
of each model:

| Model | best configuration | pooled @10% | per-fold mean (sd) |
|---|---|---|---|
| **(b) Logistic regression** | `C=10` | **0.410** | 0.415 (0.161) |
| (b) MaxEnt | linear + quadratic, `beta_multiplier=4` | 0.334 | 0.357 (0.127) |
| (a) One-Class SVM | `nu=0.2`, gamma 2× scale | 0.258 | 0.269 (0.153) |
| (a) PCA20 → One-Class SVM | 20 components, `nu=0.1` | 0.233 | 0.279 (0.132) |
| (a) Isolation Forest | `max_features=0.5` | 0.188 | 0.181 (0.141) |
| (a) Ensemble autoencoder + IF | — | 0.163 | 0.150 (0.145) |
| (a) Autoencoder (128–32–128) | — | 0.140 | 0.138 (0.122) |

**Criterion 1 (one-class list):** no one-class configuration reaches
0.30 (best 0.258), so the one-class side of RQ1 closes with Isolation
Forest and One-Class SVM. PCA, the autoencoder and the two ensembles
are recorded here as tested and dropped. The sweep moved the one-class
models little: One-Class SVM 0.230 → 0.258, Isolation Forest
0.165 → 0.188; the autoencoder (0.140) is a bit above its linear form
(PCA, 0.120) and still near chance; the autoencoder + Isolation Forest
ensemble (0.163) is no better than the Isolation Forest alone. PCA to
20 dimensions before the One-Class SVM did not help (0.233).

**Criterion 2 (classifier of the curve):** the **logistic regression**
(`C=10`, 0.410) beats the best MaxEnt (0.334) by 0.076, well outside
the 0.02 margin. It is the classifier of the RQ1 curve, with `C=10`.
Per fold it is ahead of MaxEnt in 9 blocks, behind in 5, tied in 1;
MaxEnt has the lower spread (sd 0.127 vs. 0.161).

**Criterion 3 (family):** every (b) configuration except the two
weakest MaxEnt ones (linear + quadratic with `beta_multiplier` 0.5 and
1.5, 0.239) beats every (a) configuration at 10% of area; the best
configurations of (b) beat the best of (a) at every area fraction.

**Reading of the sweep.** For the logistic regression the number rises
monotonically with weaker regularisation (`C` 0.01 → 10: 0.326, 0.348,
0.365, 0.410), and `C=10` is the edge of the grid, so a larger `C`
might go higher still; not explored, because 10% at 15–40 points per
block is already coarse and the choice is made on the test folds.
MaxEnt gained from quadratic features only under strong
regularisation (`beta_multiplier=4`: 0.334); with weaker regularisation
the quadratic features hurt (0.239 at 10%, although 0.355 per-fold
mean — the pooled number is dominated by the two large blocks).

**Caveats:** every "best" above was chosen on the same test folds that
report it (declared in section 2), for both families, so the sweep
numbers are optimistic by the same selection effect; the comparison
between models is what is read, not the absolute level. The area
sample is 15–40 background points per block (about 3% resolution per
presence, smoothed by pooling 2 464 presences). The `C=10` choice
should be confirmed on the expert arm or the curve before being
treated as more than "weaker regularisation helps here". Record and
open points:
[`thesis/notes.md`](../../thesis/notes.md#rq1-presence-only).
